In [1]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
import torch

import sys, os
# This is not super pretty, but I think this is the best way to import stuff from ../../../util?
CODE_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if CODE_ROOT not in sys.path:
    sys.path.insert(1, CODE_ROOT)

from inference import initialize_trafo_from_saved_state, get_sdss_spectra_for_inference
from train_model import eval_model, collect_dataloaders

In [2]:
config_path = f"../log_files/realistic_noise_model_snr5_config.yaml"
weights_path = f"../model_states/realistic_noise_model_snr5_weights.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, criterion, optimizer = initialize_trafo_from_saved_state(config_path, 403, 4, weights_path, device)

torch.save(model, "model.pt")

In [3]:
print("Number of network parameters:", np.sum([np.prod(theta.shape) for theta in model.parameters()]))

Number of network parameters: 3785548


In [10]:
from torchviz import make_dot

In [20]:
cat_path = "../SDSS_support_files/Custom_cat.npz"
resid_file = "../SDSS_support_files/residcorr_v5_4_45.dat"
snr_filter = 20

specs = get_sdss_spectra_for_inference(cat_path, resid_file, snr_filter)

y = model(specs)

30


  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:00<00:00, 159.47it/s]


In [2]:
models_to_plot = ["sweep_model_2", "sweep_model_10"]

In [3]:
for model_name in models_to_plot:

    print(f"Modle {model_name}")

    weights_path = f"../model_states/{model_name}_weights.pt"
    config_path = f"../log_files/{model_name}_config.yaml"

    print("Collecting dataset")

    _, _, test_loader, y_mean, y_std = collect_dataloaders(config_path)

    len_in = test_loader.dataset.X.shape[1]
    len_out = test_loader.dataset.y.shape[1]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Initalizing model")

    model, criterion, optimizer = initialize_trafo_from_saved_state(config_path, len_in, len_out, weights_path, device)

    print("Evaluating model")

    _, all_targets, all_preds = eval_model(model, test_loader, criterion, device, return_predictions=True)

    print(all_targets.shape, all_preds.shape)

Modle sweep_model_2
Initalizing model
Evaluating model


/u/jerbo/CLIMB-Project/env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 